# Week 5: Data Visualization and Communication
**Dataset:** Diamonds (53,940 rows) — carat, cut, colour, clarity, dimensions and price.
**Story:** What really drives diamond price — and a Simpson's-paradox-style surprise along the way.
Run top to bottom to regenerate every figure used in the Word report, plus the interactive Plotly chart.

In [1]:
import json, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', font_scale=1.0)
PAL = sns.color_palette('viridis', 8)
df = sns.load_dataset('diamonds').copy()
cut_order = ['Fair','Good','Very Good','Premium','Ideal']
clarity_order = ['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']
color_order = sorted(df['color'].unique())
df['cut'] = pd.Categorical(df['cut'], cut_order, ordered=True)
df['clarity'] = pd.Categorical(df['clarity'], clarity_order, ordered=True)
df['price_per_carat'] = df['price']/df['carat']
R = {}
R['n_rows'], R['n_cols'] = df.shape
R['price_min'], R['price_max'], R['price_mean'], R['price_median'] = int(df.price.min()), int(df.price.max()), round(df.price.mean(),1), int(df.price.median())
R['carat_mean'] = round(df.carat.mean(),3)

# FIG 1: Price distribution
fig, ax = plt.subplots(1,2, figsize=(11,4.3))
sns.histplot(df.price, bins=60, color=PAL[4], ax=ax[0], kde=True)
ax[0].set_title('Distribution of diamond price'); ax[0].set_xlabel('Price (US$)')
sns.histplot(np.log10(df.price), bins=60, color=PAL[2], ax=ax[1])
ax[1].set_title('Distribution of log10(price)'); ax[1].set_xlabel('log10(Price)')
plt.tight_layout(); plt.savefig('fig1_price_dist.png', dpi=160); plt.close()
R['price_skew'] = round(float(df.price.skew()),2)

# FIG 2: Carat vs Price scatter colored by cut
fig, ax = plt.subplots(figsize=(8.5,5.5))
sns.scatterplot(data=df.sample(6000, random_state=42), x='carat', y='price', hue='cut', hue_order=cut_order,
                 palette='viridis', alpha=0.5, s=18, ax=ax, linewidth=0)
ax.set_title('Carat vs Price, coloured by cut quality (6,000-point sample)')
ax.set_xlabel('Carat'); ax.set_ylabel('Price (US$)')
ax.legend(title='Cut', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout(); plt.savefig('fig2_carat_price.png', dpi=160); plt.close()
R['carat_price_corr'] = round(float(df.carat.corr(df.price)),3)

# FIG 3: Price by cut and clarity boxplots
fig, ax = plt.subplots(1,2, figsize=(12,4.6))
sns.boxplot(data=df, x='cut', y='price', order=cut_order, palette='viridis', ax=ax[0], showfliers=False)
ax[0].set_title('Price by cut quality'); ax[0].set_xlabel('Cut'); ax[0].set_ylabel('Price (US$)')
ax[0].tick_params(axis='x', rotation=15)
sns.boxplot(data=df, x='clarity', y='price', order=clarity_order, palette='viridis', ax=ax[1], showfliers=False)
ax[1].set_title('Price by clarity grade'); ax[1].set_xlabel('Clarity'); ax[1].set_ylabel('Price (US$)')
plt.tight_layout(); plt.savefig('fig3_box_cut_clarity.png', dpi=160); plt.close()
R['median_price_fair'] = int(df[df.cut=='Fair'].price.median())
R['median_price_ideal'] = int(df[df.cut=='Ideal'].price.median())
R['median_price_i1'] = int(df[df.clarity=='I1'].price.median())
R['median_price_if'] = int(df[df.clarity=='IF'].price.median())

# FIG 4: Correlation heatmap
num = df[['carat','depth','table','price','x','y','z']]
corr = num.corr()
plt.figure(figsize=(6.5,5.3))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, cbar_kws={'shrink':0.8})
plt.title('Correlation matrix of numeric diamond attributes')
plt.tight_layout(); plt.savefig('fig4_heatmap.png', dpi=160); plt.close()
R['depth_price_corr'] = round(float(corr.loc['depth','price']),3)
R['table_price_corr'] = round(float(corr.loc['table','price']),3)

# FIG 5: Average price per carat by color, grouped by clarity (grouped bar)
grp = df.groupby(['color','clarity'], observed=True)['price_per_carat'].mean().reset_index()
piv = grp.pivot(index='color', columns='clarity', values='price_per_carat').reindex(index=color_order, columns=clarity_order)
fig, ax = plt.subplots(figsize=(10,5))
piv.plot(kind='bar', ax=ax, colormap='viridis', width=0.82)
ax.set_title('Average price-per-carat by colour grade, split by clarity')
ax.set_xlabel('Colour grade (D = best, J = worst)'); ax.set_ylabel('Avg. price per carat (US$)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Clarity', bbox_to_anchor=(1.02,1), loc='upper left', ncol=1, fontsize=8)
plt.tight_layout(); plt.savefig('fig5_color_clarity_bar.png', dpi=160); plt.close()
R['best_color'] = piv.mean(axis=1).idxmax(); R['worst_color'] = piv.mean(axis=1).idxmin()

# FIG 6: Carat distribution by cut (violin) + trend line price vs carat (regplot) combined
fig, ax = plt.subplots(1,2, figsize=(12,4.8))
sns.violinplot(data=df, x='cut', y='carat', order=cut_order, palette='viridis', ax=ax[0], cut=0)
ax[0].set_title('Carat size distribution by cut quality'); ax[0].tick_params(axis='x', rotation=15)
sample = df.sample(4000, random_state=1)
sns.regplot(data=sample, x='carat', y='price', scatter_kws={'alpha':0.25,'s':14,'color':PAL[3]},
            line_kws={'color':'crimson'}, lowess=True, ax=ax[1])
ax[1].set_title('Price vs carat with LOWESS trend (non-linear)')
plt.tight_layout(); plt.savefig('fig6_violin_trend.png', dpi=160); plt.close()

# FIG 7: Price-per-carat trend across carat bins (line) - "peculiarity": diminishing/accelerating returns
bins = pd.cut(df.carat, bins=[0,0.3,0.5,0.7,1.0,1.5,2.0,3.0,5.5])
tr = df.groupby(bins, observed=True)['price_per_carat'].mean().reset_index()
tr['carat_bin'] = tr['carat'].astype(str)
fig, ax = plt.subplots(figsize=(9,4.5))
sns.lineplot(data=tr, x='carat_bin', y='price_per_carat', marker='o', color=PAL[5], ax=ax, linewidth=2.5)
ax.set_title('Average price per carat rises sharply with carat size')
ax.set_xlabel('Carat range'); ax.set_ylabel('Avg. price per carat (US$)')
for x,y in zip(tr['carat_bin'], tr['price_per_carat']):
    ax.annotate(f'${y:,.0f}', (x,y), textcoords='offset points', xytext=(0,8), ha='center', fontsize=8)
plt.tight_layout(); plt.savefig('fig7_ppc_trend.png', dpi=160); plt.close()
R['ppc_small'] = round(float(tr.price_per_carat.iloc[0]),0)
R['ppc_large'] = round(float(tr.price_per_carat.iloc[-1]),0)

# FIG 8 (interactive): Plotly scatter carat vs price, color=clarity, size=depth, facet by cut
samp = df.sample(3000, random_state=7)
figp = px.scatter(samp, x='carat', y='price', color='clarity', category_orders={'clarity':clarity_order,'cut':cut_order},
                   facet_col='cut', facet_col_wrap=3, opacity=0.55, color_discrete_sequence=px.colors.sequential.Viridis_r,
                   title='Interactive: Carat vs Price by Cut and Clarity (hover for details)',
                   labels={'carat':'Carat','price':'Price (US$)'}, height=750, width=1150)
figp.update_layout(template='plotly_white', legend_title='Clarity')
figp.write_html('fig8_interactive_scatter.html', include_plotlyjs='cdn')

json.dump(R, open('viz_results.json','w'), indent=2, default=str)
print(R)

# Static equivalent of the interactive chart for the printed report
fig, axes = plt.subplots(2,3, figsize=(13,7.5), sharex=True, sharey=True)
for ax, ct in zip(axes.flat, cut_order):
    sub = samp[samp.cut==ct]
    sc = ax.scatter(sub.carat, sub.price, c=sub.clarity.cat.codes, cmap='viridis', s=14, alpha=0.6)
    ax.set_title(ct)
fig.suptitle('Carat vs Price by Cut, coloured by Clarity (static view of the interactive chart)')
for ax in axes[-1,:]: ax.set_xlabel('Carat')
for ax in axes[:,0]: ax.set_ylabel('Price (US$)')
cbar = fig.colorbar(sc, ax=axes, shrink=0.6, ticks=range(len(clarity_order)))
cbar.ax.set_yticklabels(clarity_order); cbar.set_label('Clarity')
plt.savefig('fig8_static.png', dpi=150, bbox_inches='tight'); plt.close()


/tmp/ipykernel_179/452275357.py:42: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='cut', y='price', order=cut_order, palette='viridis', ax=ax[0], showfliers=False)
/tmp/ipykernel_179/452275357.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='clarity', y='price', order=clarity_order, palette='viridis', ax=ax[1], showfliers=False)


/tmp/ipykernel_179/452275357.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x='cut', y='carat', order=cut_order, palette='viridis', ax=ax[0], cut=0)


{'n_rows': 53940, 'n_cols': 11, 'price_min': 326, 'price_max': 18823, 'price_mean': np.float64(3932.8), 'price_median': 2401, 'carat_mean': np.float64(0.798), 'price_skew': 1.62, 'carat_price_corr': 0.922, 'median_price_fair': 3282, 'median_price_ideal': 1810, 'median_price_i1': 3344, 'median_price_if': 1080, 'depth_price_corr': -0.011, 'table_price_corr': 0.127, 'best_color': 'D', 'worst_color': 'I', 'ppc_small': 2209.0, 'ppc_large': 4615.0}


## Summary of findings
- Carat (size) correlates with price at r ≈ 0.92 — by far the dominant driver.
- Raw price comparisons by cut/clarity are misleading: Fair cut has a *higher* median price than Ideal cut.
- The cause: Fair-cut stones average larger carat weight than Ideal-cut stones (cutters sacrifice cut precision on bigger rough stones).
- Once normalised to price-per-carat, quality grades behave as expected: better colour/clarity → higher price per carat.
- Price per carat itself accelerates with carat size — large stones carry a rising premium, not just a proportional one.